# USING THE MODEL WITHOUT FINE TUNING

# 0. Install dependencies

# 1. Import and Load Model

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/pegasus-xsum"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

c:\Users\Administrator\anaconda3\envs\pg\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\Administrator\anaconda3\envs\pg\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
import os
import torch
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
folder_path = "C:\\Users\\Administrator\\Downloads\\Dataset\\merged_data\\"

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

In [5]:
train_df = pd.read_csv(folder_path + "train.csv")
val_df = pd.read_csv(folder_path + "val.csv")
test_df = pd.read_csv(folder_path + "test.csv")

In [34]:
def summarize_batch(texts, batch_size=4):
    summaries = []
    num_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(num_batches):
        batch_texts = texts[i*batch_size : (i+1)*batch_size]

        print(f"\n===== BATCH {i+1}/{num_batches} =====")
        print(f"Input size: {len(batch_texts)}")

        # Generate summaries
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model.generate(**inputs, max_length=256, num_beams=4)
        batch_summaries = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for j, summary in enumerate(batch_summaries):
            print(f"\n--- Sample {j+1} in batch ---")
            print("SUMMARY:", summary)

        summaries.extend(batch_summaries)

    return summaries

In [35]:
test_texts = test_df["full_story"].astype(str).tolist()

test_df["pgs_not_finetune_summary"] = summarize_batch(
    test_texts,
    batch_size=4
)


===== BATCH 1/308 =====
Input size: 4

--- Sample 1 in batch ---
SUMMARY: Scientists uncovered more than 400 genes linked to different forms of unhealthy aging, a.k.a, frailty. The findings lend support to what is known as the 'geroscience hypothesis' -- the idea that to treat the multiple chronic illnesses that come with aging, we must treat aging itself.

--- Sample 2 in batch ---
SUMMARY: The origin of life on Earth has long been a mystery that has eluded scientists. A key question is how much of the history of life on Earth is lost to time. It is quite common for a single species to "phase out" using a biochemical reaction, and if this happens across enough species, such reactions could effectively be "forgotten" by life on Earth.

--- Sample 3 in batch ---
SUMMARY: A study reveals that inflammation associated with Marfan syndrome increases vulnerability to neurological diseases and complications following strokes, as demonstrated in animal models.

--- Sample 4 in batch ---
SUMMA

In [36]:
import numpy as np
from evaluate import load
from bert_score import score
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Model gốc (chưa fine-tune)
model_name = "google/pegasus-xsum"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()
model.to("cuda")

rouge = load("rouge")

num_samples = 200
preds = []
refs = []

print("Đang generate bằng Pegasus-XSUM gốc (chưa fine-tune)...")
with torch.no_grad():
    for i in tqdm(range(num_samples)):
        item = test_dataset[i]
        input_ids = item["input_ids"].unsqueeze(0).cuda()

        gen_ids = model.generate(
            input_ids,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )
        pred = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
        preds.append(pred)

        label = np.where(item["labels"] == -100, tokenizer.pad_token_id, item["labels"])
        ref = tokenizer.decode(label, skip_special_tokens=True)
        refs.append(ref)

result = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
P, R, F = score(preds, refs, lang="en", verbose=False)

print("\n=== PEGASUS-XSUM GỐC (CHƯA FINE-TUNE) ===")
print(f"ROUGE-1: {result['rouge1']*100:.3f}")
print(f"ROUGE-2: {result['rouge2']*100:.3f}")
print(f"ROUGE-L: {result['rougeL']*100:.3f}")
print(f"BERTScore: {F.mean().item()*100:.3f}")

c:\Users\Administrator\anaconda3\envs\pg\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Đang generate bằng Pegasus-XSUM gốc (chưa fine-tune)...


100%|██████████| 200/200 [02:40<00:00,  1.25it/s]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== PEGASUS-XSUM GỐC (CHƯA FINE-TUNE) ===
ROUGE-1: 25.331
ROUGE-2: 7.907
ROUGE-L: 18.429
BERTScore: 87.128


# USING THE MODEL WITH FINE TUNING

In [38]:
from transformers import PegasusTokenizer, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Trainer
from transformers import Seq2SeqTrainer
from evaluate import load
import numpy as np
import torch
from bert_score import score
from torch.utils.data import DataLoader, Dataset

In [39]:
max_input_length = 512
max_target_length = 128

def chunk_text(tokens, chunk_size=512):
    return [tokens[i:i + chunk_size] for i in range(0, len(tokens), chunk_size)]

class SummarizationDataset(Dataset):
    def __init__(self, df, tokenizer, max_input=512, max_target=128):
        self.tokenizer = tokenizer
        self.max_input = max_input
        self.max_target = max_target
        self.inputs = []
        self.labels = []

        for story, summary in zip(df['full_story'], df['abstract']):
            # Tokenize story (không truncate để chunk)
            story_encoding = tokenizer(
                story,
                truncation=False,
                return_tensors="pt"
            )
            story_ids = story_encoding.input_ids[0].tolist()

            # Chunk story nếu quá dài
            chunks = chunk_text(story_ids, max_input)

            # Tokenize summary
            summary_encoding = tokenizer(
                summary,
                max_length=max_target,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )
            labels = summary_encoding.input_ids[0]
            # Thay pad_token_id bằng -100 để loss bỏ qua
            labels = torch.where(labels == tokenizer.pad_token_id, -100, labels)

            # Lặp qua từng chunk
            for chunk in chunks:
                # Pad chunk nếu cần
                if len(chunk) < max_input:
                    chunk = chunk + [tokenizer.pad_token_id] * (max_input - len(chunk))
                else:
                    chunk = chunk[:max_input]  # đảm bảo không vượt

                self.inputs.append(torch.tensor(chunk, dtype=torch.long))
                self.labels.append(labels.clone())  # clone vì sẽ dùng lại

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return {
            "input_ids": self.inputs[idx],
            "attention_mask": (self.inputs[idx] != tokenizer.pad_token_id).long(),
            "labels": self.labels[idx]
        }

# Tạo dataset
train_dataset = SummarizationDataset(train_df, tokenizer, max_input_length, max_target_length)
val_dataset   = SummarizationDataset(val_df,   tokenizer, max_input_length, max_target_length)
test_dataset  = SummarizationDataset(test_df,  tokenizer, max_input_length, max_target_length)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=0)

Token indices sequence length is longer than the specified maximum sequence length for this model (735 > 512). Running this sequence through the model will result in indexing errors


In [40]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model, 
    padding="longest",           
    max_length=max_input_length, 
    pad_to_multiple_of=8,          
    return_tensors="pt"
)

In [9]:
rouge = load("rouge")
bert_scorer = lambda cands, refs: score(
    cands,
    refs,
    lang="en",
    verbose=False,
    model_type="microsoft/deberta-xlarge-mnli"
)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    P, R, F = bert_scorer(decoded_preds, decoded_labels)

    return {
        "rouge1_f": rouge_result["rouge1"].mid.fmeasure,
        "rouge2_f": rouge_result["rouge2"].mid.fmeasure,
        "rougeL_f": rouge_result["rougeL"].mid.fmeasure,
        "bertscore_f": float(F.mean()),
    }

In [41]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./pegasus-finetuned",
    overwrite_output_dir=True,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    evaluation_strategy="steps",
    eval_steps=2000,
    save_steps=2000,
    logging_steps=100,
    save_total_limit=3,

    learning_rate=3e-5,
    warmup_steps=500,
    weight_decay=0.01,
    num_train_epochs=4,

    fp16=True,

    dataloader_num_workers=0,
    dataloader_pin_memory=True,

    load_best_model_at_end=True,
    metric_for_best_model="rouge2",
    greater_is_better=True,

    report_to="none",


    remove_unused_columns=False,
    group_by_length=False,
)

In [42]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [43]:
trainer.train()

  0%|          | 0/2612 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 24.00 GiB of which 0 bytes is free. Of the allocated memory 20.03 GiB is allocated by PyTorch, and 33.59 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [13]:
trainer.save_model("./pegasus_finetuned")
tokenizer.save_pretrained("./pegasus_finetuned")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 64, 'num_beams': 8, 'length_penalty': 0.6, 'forced_eos_token_id': 1}


('./pegasus_finetuned\\tokenizer_config.json',
 './pegasus_finetuned\\special_tokens_map.json',
 './pegasus_finetuned\\spiece.model',
 './pegasus_finetuned\\added_tokens.json',
 './pegasus_finetuned\\tokenizer.json')

In [32]:
import numpy as np
from evaluate import load
from bert_score import score
from tqdm import tqdm
import torch

rouge = load("rouge")

num_samples = 200
model.eval()
preds = []
refs = []

with torch.no_grad():
    for i in tqdm(range(num_samples)):
        item = test_dataset[i]
        input_ids = item["input_ids"].unsqueeze(0).cuda()

        gen_ids = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
        pred = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
        preds.append(pred)

        label = np.where(item["labels"] == -100, tokenizer.pad_token_id, item["labels"])
        ref = tokenizer.decode(label, skip_special_tokens=True)
        refs.append(ref)

result = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
P, R, F = score(preds, refs, lang="en", verbose=False)

print(f"ROUGE-1: {result['rouge1']*100:.3f}")
print(f"ROUGE-2: {result['rouge2']*100:.3f}")
print(f"ROUGE-L: {result['rougeL']*100:.3f}")
print(f"BERTScore: {F.mean().item()*100:.3f}")

100%|██████████| 200/200 [05:43<00:00,  1.72s/it]
c:\Users\Administrator\anaconda3\envs\pg\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE-1: 37.926
ROUGE-2: 21.892
ROUGE-L: 31.557
BERTScore: 89.698
